##Triển khai
Bây giờ chúng ta hãy cùng triển khai mô hình GRU đơn giản trong Python. Chúng ta sẽ bắt đầu bằng việc chuẩn bị các thư viện và tập dữ liệu cần thiết.

In [1]:
#1. Nhập thư viện
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

## 2. Tải tập dữ liệu
Bộ dữ liệu chúng tôi đang sử dụng là bộ dữ liệu chuỗi thời gian chứa dữ liệu nhiệt độ hàng ngày, tức là bộ dữ liệu dự báo. Nó bao gồm 8.000 ngày, bắt đầu từ ngày 1 tháng 1 năm 2010.

In [3]:
df = pd.read_csv('/content/data (1).csv', parse_dates=['Date'], index_col='Date')
print(df.head())

            Temperature
Date                   
2010-01-01    27.483571
2010-01-02    24.308678
2010-01-03    28.238443
2010-01-04    32.615149
2010-01-05    23.829233


##3. Xử lý sơ bộ dữ liệu
Dữ liệu được chuẩn hóa bằng MinMaxScaler để đưa các giá trị đặc trưng về khoảng từ 0 đến 1. Chuẩn hóa giúp mạng nơ-ron huấn luyện hiệu quả hơn và ngăn ngừa sai lệch do các đặc trưng có giá trị lớn hơn gây ra.

* Sử dụng MinMaxScaler để chuẩn hóa.
* Thang đo các đặc điểm trong phạm vi 0–1
* Cải thiện hiệu suất huấn luyện mạng nơ-ron.
* Ngăn chặn sự chi phối của các giá trị đặc trưng lớn hơn

In [4]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df.values)

## 4. Chuẩn bị dữ liệu cho GRU
Chúng ta sẽ định nghĩa một hàm để chuẩn bị dữ liệu cho việc huấn luyện mô hình.

* create_dataset(): Chuẩn bị tập dữ liệu cho dự báo chuỗi thời gian. Nó tạo ra các cửa sổ trượt có độ dài time_step để dự đoán bước thời gian tiếp theo.
* X.reshape() : Định hình lại dữ liệu đầu vào để phù hợp với hình dạng mong muốn của GRU là 3D: tức là các mẫu, bước thời gian và đặc trưng.

In [5]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

## 5. Xây dựng mô hình GRU
Chúng ta sẽ định nghĩa mô hình GRU của mình với các thành phần sau:

* GRU(units=50): Thêm một lớp GRU với 50 đơn vị (nơron).
* return_sequences=True: Đảm bảo lớp GRU trả về toàn bộ chuỗi (cần thiết để xếp chồng nhiều lớp GRU).
* Dense(units=1): Lớp đầu ra dự đoán một giá trị duy nhất cho bước thời gian tiếp theo.
* Adam(): Một thuật toán tối ưu hóa thích ứng thường được sử dụng trong học sâu.

In [6]:
model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## 6. Huấn luyện mô hình

Phương thức `model.fit()` huấn luyện mô hình trên tập dữ liệu đã chuẩn bị. Tham số `epochs=10` chỉ định số lần lặp trên toàn bộ tập dữ liệu, và `batch_size=32` xác định số lượng mẫu trên mỗi lô.

In [7]:
model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 32s 116ms/step - loss: 0.0218
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 44s 130ms/step - loss: 0.0182
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 28s 112ms/step - loss: 0.0180
Epoch 4/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 40s 108ms/step - loss: 0.0181
Epoch 5/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 109ms/step - loss: 0.0178
Epoch 6/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 28s 113ms/step - loss: 0.0178
Epoch 7/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 28s 113ms/step - loss: 0.0177
Epoch 8/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 110ms/step - loss: 0.0177
Epoch 9/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 109ms/step - loss: 0.0178
Epoch 10/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 41s 109ms/step - loss: 0.0178


## 7. Đưa ra dự đoán
Mô hình GRU đã được huấn luyện được sử dụng để dự đoán các giá trị trong tương lai từ chuỗi dữ liệu đầu vào.

* Sử dụng 100 giá trị nhiệt độ đã được hiệu chỉnh gần nhất làm đầu vào.
* Điều chỉnh đầu vào thành (1, time_step, 1) để tương thích với GRU.
* samples = 1, time_steps = 100, and features = 1
* model.predict() tạo ra các dự đoán từ mô hình đã được huấn luyện.

In [8]:
input_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step


## 8. Biến đổi ngược các dự đoán
Biến đổi ngược các dự đoán đề cập đến quá trình chuyển đổi các dự đoán đã được chuẩn hóa (tỷ lệ hóa) trở lại tỷ lệ ban đầu của chúng.

* scaler.inverse_transform(): Chuyển đổi các dự đoán đã chuẩn hóa trở lại thang đo ban đầu.

In [9]:
predicted_values = scaler.inverse_transform(predicted_values)
print(
    f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

The predicted temperature for the next day is: 24.98°C
